In [1]:
import numpy as np

In [2]:
class DataAnalytics:
    """Encapsulates all NumPy functionality."""

    def __init__(self, array=None):                      # constructor
        self.__array = None if array is None else np.array(array)

    # ---------- private methods (internal computations) ----------
    def __require_array(self):
        if self.__array is None:
            raise ValueError("No array yet. Create one first (option 1).")

    def __to_same_shape(self, values):
        values = np.array(values)
        if values.size != self.__array.size:
            raise ValueError(f"Need exactly {self.__array.size} elements.")
        return values.reshape(self.__array.shape)

    # ---------- class-level and static methods ----------
    @classmethod
    def from_values(cls, values, shape):
        return cls(np.array(values).reshape(shape))

    @staticmethod
    def parse_numbers(text):
        return [float(x) if "." in x else int(x) for x in text.split()]

    # ---------- array management ----------
    @property
    def array(self):
        self.__require_array()
        return self.__array

    def create(self, values, shape):
        values = np.array(values)
        if values.size != int(np.prod(shape)):
            raise ValueError(f"Need exactly {int(np.prod(shape))} elements.")
        self.__array = values.reshape(shape)
        return self.__array

    def index(self, idx):
        self.__require_array()
        return self.__array[idx]

    def slice(self, slices):
        self.__require_array()
        return self.__array[tuple(slices)]

    def combine(self, values, axis=0):
        self.__require_array()
        return np.concatenate((self.__array, self.__to_same_shape(values)), axis=axis)

    def split(self, parts, axis=0):
        self.__require_array()
        return np.array_split(self.__array, parts, axis=axis)

    # ---------- math ----------
    def arithmetic(self, op, values):
        self.__require_array()
        other = self.__to_same_shape(values)
        ops = {"1": np.add, "2": np.subtract, "3": np.multiply, "4": np.divide}
        return ops[op](self.__array, other)

    def dot_product(self, values):
        self.__require_array()
        return np.dot(self.__array.flatten(), self.__to_same_shape(values).flatten())

    def matrix_multiply(self, values, shape):
        self.__require_array()
        other = np.array(values).reshape(shape)
        return np.matmul(self.__array, other)

    # ---------- search, sort, filter ----------
    def search(self, value):
        self.__require_array()
        return np.argwhere(self.__array == value)

    def sort(self, descending=False):
        self.__require_array()
        result = np.sort(self.__array, axis=-1)          # row-wise
        return result[..., ::-1] if descending else result

    def filter(self, operator, value):
        self.__require_array()
        ops = {">": np.greater, "<": np.less, ">=": np.greater_equal,
               "<=": np.less_equal, "==": np.equal, "!=": np.not_equal}
        return self.__array[ops[operator](self.__array, value)]

    # ---------- aggregates and statistics ----------
    def aggregate(self, name):
        self.__require_array()
        funcs = {"Sum": np.sum, "Mean": np.mean, "Median": np.median,
                 "Standard Deviation": np.std, "Variance": np.var,
                 "Minimum": np.min, "Maximum": np.max}
        return funcs[name](self.__array)

    def percentile(self, q):
        self.__require_array()
        return np.percentile(self.__array, q)

    def correlation(self, values):
        self.__require_array()
        other = self.__to_same_shape(values)
        return np.corrcoef(self.__array.flatten(), other.flatten())[0, 1]

In [3]:
def read_int(prompt):
    return int(input(prompt).strip())

def read_numbers(prompt, count):
    nums = DataAnalytics.parse_numbers(input(prompt))
    if len(nums) != count:
        raise ValueError(f"Expected {count} elements, got {len(nums)}.")
    return nums

def parse_range(text):
    start, end = text.split(":")
    return slice(int(start) if start else None, int(end) if end else None)

def show(title, arr):
    print(f"\n{title}\n{arr}")

In [4]:
def create_menu(app):
    print("\nSelect the type of array to create:\n1. 1D Array\n2. 2D Array\n3. 3D Array")
    kind = input("Enter your choice: ").strip()
    if kind == "1":
        shape = (read_int("\nEnter the number of elements: "),)
    elif kind == "2":
        r = read_int("\nEnter the number of rows: ")
        c = read_int("Enter the number of columns: ")
        shape = (r, c)
    elif kind == "3":
        d = read_int("\nEnter the number of blocks (depth): ")
        r = read_int("Enter the number of rows: ")
        c = read_int("Enter the number of columns: ")
        shape = (d, r, c)
    else:
        raise ValueError("Invalid array type.")
    size = int(np.prod(shape))
    values = read_numbers(f"Enter {size} elements for the array separated by space: ", size)
    show("Array created successfully:", app.create(values, shape))
    access_menu(app)


def access_menu(app):
    names = {1: ["range"], 2: ["row", "column"], 3: ["depth", "row", "column"]}
    while True:
        print("\nChoose an operation:\n1. Indexing\n2. Slicing\n3. Go Back")
        c = input("Enter your choice: ").strip()
        if c == "1":
            idx = tuple(int(i) for i in input("\nEnter index (e.g. 1,2): ").split(","))
            print("\nElement:", app.index(idx))
        elif c == "2":
            slices = [parse_range(input(f"Enter the {n} range (start:end): "))
                      for n in names[app.array.ndim]]
            show("Sliced Array:", app.slice(slices))
        elif c == "3":
            break


def math_menu(app):
    print("\nChoose a mathematical operation:\n1. Addition\n2. Subtraction"
          "\n3. Multiplication\n4. Division\n5. Dot Product\n6. Matrix Multiplication")
    c = input("Enter your choice: ").strip()
    size = app.array.size
    if c == "6":
        r = read_int("\nEnter rows of second matrix: ")
        cols = read_int("Enter columns of second matrix: ")
        vals = read_numbers(f"Enter {r*cols} elements separated by space: ", r * cols)
        show("Original Array:", app.array)
        show("Result of Matrix Multiplication:", app.matrix_multiply(vals, (r, cols)))
        return
    vals = read_numbers(f"\nEnter the same-size array elements ({size} elements separated by space): ", size)
    show("Original Array:", app.array)
    show("Second Array:", np.array(vals).reshape(app.array.shape))
    if c in "1234" and c != "":
        names = {"1": "Addition", "2": "Subtraction", "3": "Multiplication", "4": "Division"}
        show(f"Result of {names[c]}:", app.arithmetic(c, vals))
    elif c == "5":
        show("Result of Dot Product:", app.dot_product(vals))
    else:
        raise ValueError("Invalid operation.")


def combine_split_menu(app):
    print("\nChoose an option:\n1. Combine Arrays\n2. Split Array")
    c = input("Enter your choice: ").strip()
    if c == "1":
        size = app.array.size
        vals = read_numbers(f"\nEnter the elements of another array to combine ({size} elements separated by space): ", size)
        show("Original Array:", app.array)
        show("Second Array:", np.array(vals).reshape(app.array.shape))
        show("Combined Array (Vertical Stack):", app.combine(vals, axis=0))
    elif c == "2":
        parts = read_int("\nHow many parts to split into? ")
        show("Original Array:", app.array)
        for i, part in enumerate(app.split(parts, axis=0), 1):
            show(f"Part {i}:", part)
    else:
        raise ValueError("Invalid option.")


def search_sort_filter_menu(app):
    print("\nChoose an option:\n1. Search a value\n2. Sort the array\n3. Filter values")
    c = input("Enter your choice: ").strip()
    show("Original Array:", app.array)
    if c == "1":
        v = float(input("\nEnter the value to search: "))
        pos = app.search(v)
        print("Found at positions:", pos.tolist() if len(pos) else "Value not found")
    elif c == "2":
        desc = input("\nSort descending? (y/n): ").strip().lower() == "y"
        show("Sorted Array:", app.sort(desc))
        print("(Sorting applied row-wise.)")
    elif c == "3":
        op = input("\nEnter condition operator (>, <, >=, <=, ==, !=): ").strip()
        v = float(input("Enter the value: "))
        show("Filtered Values:", app.filter(op, v))
    else:
        raise ValueError("Invalid option.")


def stats_menu(app):
    print("\nChoose an aggregate/statistical operation:\n1. Sum\n2. Mean\n3. Median"
          "\n4. Standard Deviation\n5. Variance\n6. Minimum\n7. Maximum"
          "\n8. Percentile\n9. Correlation Coefficient")
    c = input("Enter your choice: ").strip()
    names = {"1": "Sum", "2": "Mean", "3": "Median", "4": "Standard Deviation",
             "5": "Variance", "6": "Minimum", "7": "Maximum"}
    show("Original Array:", app.array)
    if c in names:
        print(f"\n{names[c]} of Array: {app.aggregate(names[c])}")
    elif c == "8":
        q = float(input("\nEnter percentile (0-100): "))
        print(f"\n{q}th Percentile: {app.percentile(q)}")
    elif c == "9":
        size = app.array.size
        vals = read_numbers(f"\nEnter second array ({size} elements separated by space): ", size)
        print(f"\nCorrelation Coefficient: {app.correlation(vals)}")
    else:
        raise ValueError("Invalid option.")

In [5]:
def main():
    app = DataAnalytics()
    print("Welcome to the NumPy Analyzer!")
    print("=" * 40)
    while True:
        print("\nChoose an option:\n1. Create a Numpy Array\n2. Perform Mathematical Operations"
              "\n3. Combine or Split Arrays\n4. Search, Sort, or Filter Arrays"
              "\n5. Compute Aggregates and Statistics\n6. Exit")
        choice = input("Enter your choice: ").strip()
        try:
            if choice == "1":   create_menu(app)
            elif choice == "2": math_menu(app)
            elif choice == "3": combine_split_menu(app)
            elif choice == "4": search_sort_filter_menu(app)
            elif choice == "5": stats_menu(app)
            elif choice == "6":
                print("\nThank you for using the NumPy Analyzer! Goodbye!")
                break
            else:
                print("Invalid choice. Please enter 1-6.")
        except (ValueError, KeyError, IndexError) as e:
            print(f"\nError: {e}")

main()

Welcome to the NumPy Analyzer!

Choose an option:
1. Create a Numpy Array
2. Perform Mathematical Operations
3. Combine or Split Arrays
4. Search, Sort, or Filter Arrays
5. Compute Aggregates and Statistics
6. Exit


Enter your choice:  1



Select the type of array to create:
1. 1D Array
2. 2D Array
3. 3D Array


Enter your choice:  1

Enter the number of elements:  3
Enter 3 elements for the array separated by space:  10 20 30



Array created successfully:
[10 20 30]

Choose an operation:
1. Indexing
2. Slicing
3. Go Back


Enter your choice:  1

Enter index (e.g. 1,2):  2


... (19 outputs left out)

Enter your choice:  1



Original Array:
[10 20 30]

Sum of Array: 60

Choose an option:
1. Create a Numpy Array
2. Perform Mathematical Operations
3. Combine or Split Arrays
4. Search, Sort, or Filter Arrays
5. Compute Aggregates and Statistics
6. Exit


Enter your choice:  6



Thank you for using the NumPy Analyzer! Goodbye!
